# Stalled Corners in Bin-Center Redshifts ([#165](https://github.com/nanograv/holodeck/issues/165))

`sam_cyutils.dynamic_binary_number_at_fobs` gives each SAM grid corner its final redshift, and flags a corner whose binary stalls before reaching the target frequency by $z = 0$ with `redz_final = -1`. Each $(M, q, z)$ cell's redshift used to be the plain mean of its 8 corners with the $-1$ included. A cell with one stalled corner could then land at a tiny positive redshift, which placed its binaries at kpc–Mpc distances. With BIO hardening, stalling depends on $M$ and $q$, so such cells are common.

`holodeck.gravwaves.redz_at_bin_centers` now does this average for every call site, by one of three methods:
1. **`'weighted'`** (default, `holodeck.gravwaves.REDZ_CENTERS_METHOD`): the mean over corners that didn't stall, weighted by `diff_num` (consistent with the trapezoid weights of `integrate_differential_number_3dx1d`). A cell with no valid corner keeps $-1$.
2. **`'legacy'`**: the old plain mean.
3. **`'drop'`**: stalled corners set to $-\infty$ first, so any cell with a stalled corner is dropped. This is the rule `lib_tools.run_model` already applied to its GWB.

### Contents
1. The cell from the issue
2. Two models: classic phenom (`Fixed_Time_2PL_SAM`) and NG20 fiducial (BIO)
3. Where binaries are placed
4. GWB over many realizations
5. Loudest sources
6. `'legacy'` reproduces the old average
7. Summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter, ScalarFormatter
import kalepy as kale

import holodeck as holo
import holodeck.librarian
from holodeck import cosmo, utils
from holodeck import gravwaves as grav
from holodeck.constants import MPC, YR
from holodeck.sams import sam_cyutils

holo.log.setLevel("WARNING")    # holodeck logs every model build at INFO

print("Holodeck version:", holo.__version__)
print("Default method:", grav.REDZ_CENTERS_METHOD)

## 1. The Cell from the Issue

One stalled corner among seven at $z \approx 0.01$–$0.26$. Equal `dnum` makes `'weighted'` the plain mean of the seven valid corners.

In [ ]:
corners = np.reshape([-1.0, 0.008, 0.012, 0.025, 0.221, 0.241, 0.24, 0.261], (2, 2, 2, 1))
for method in ('legacy', 'weighted', 'drop'):
    z = grav.redz_at_bin_centers(corners, method=method, dnum=np.ones_like(corners)).item()
    dc = cosmo.z_to_dcom(z) / MPC if z > 0 else np.nan
    print(f"{method:>8s}:  z = {z:+.4f},  d_c = {dc:7.1f} Mpc")

## 2. Two Models

Both use the issue's frequency grid (20 yr, 30 bins) and their parameter space's default SAM shape.
- **Phenom**: `PS_Classic_Phenom_Uniform` at its defaults (`Fixed_Time_2PL_SAM`, $\tau_f$ = 3 Gyr). Stalling depends on the merger time and not on $M$, $q$.
- **NG20 fiducial**: `PS_NG20_Fiducial` (BIO, `FixedOuterTime_InnerPL_SAM`) at the issue's reproducer point.

A cell is *mixed* when some, but not all, of its corners stalled. Only mixed cells differ between methods by more than the weighting.

In [ ]:
DUR, NFREQS = 20.0 * YR, 30
fobs_cents, fobs_edges = utils.pta_freqs(dur=DUR, num=NFREQS)


def build(pspace, params):
    # population on the grid corners, as in lib_tools.run_model
    space = holo.librarian.param_spaces_dict[pspace](holo.log, nsamples=None)
    sam, hard = space.model_for_params(params)
    redz, dnum = sam_cyutils.dynamic_binary_number_at_fobs(fobs_cents / 2.0, sam, hard, cosmo)
    edges = [sam.mtot, sam.mrat, sam.redz, fobs_edges / 2.0]
    number = sam_cyutils.integrate_differential_number_3dx1d(edges, dnum)
    return dict(edges=edges, redz=redz, dnum=dnum, number=number)


models = {
    "Phenom": build('PS_Classic_Phenom_Uniform', dict(hard_time=3.0)),
    "NG20 fiducial": build('PS_NG20_Fiducial', dict(
        hard_outer_time=0.59, hard_r_gw_crit_9_log10=3.923, hard_nu_inner=3.771, mmb_zplaw_amp=1.702)),
}

for name, mod in models.items():
    # fraction of each cell's 8 corners that stalled
    frac = utils.midpoints_multiax((mod['redz'] < 0.0).astype(float), axis=(0, 1, 2))
    mixed = (frac > 0.0) & (frac < 1.0)
    num = mod['number']
    print(f"{name}: grid {mod['redz'].shape[:3]}, corners stalled {np.mean(mod['redz'] < 0):.1%}, "
          f"mixed cells {mixed.mean():.2%} holding {num[mixed].sum() / num.sum():.2%} of binaries")

## 3. Where Binaries Are Placed

Old paths fall into two groups:
- **old, $-1$ corners**: `sam.gwb_new`, `sam.gwb` / `single_sources.ss_gws_redz`, `extensions.get_samples_from_edges`, `lib_tools._calc_model_details`, and `flows_lib.run_cws`, which averaged with the $-1$ included.
- **old `run_model`**: `lib_tools.run_model`'s GWB, which set stalled corners to $-\infty$ first and so dropped every mixed cell. Its single sources still saw the $-1$ corners.

A population spread through space has $N(<d_c) \propto d_c^3$ nearby. The dashed line is $d_c^3$ through the new curve at 300 Mpc.

With `Fixed_Time_2PL_SAM`, corners stall only where the final redshift reaches $z = 0$, so the old average pulled the nearest mixed cells below zero and dropped them. With BIO, stalling depends on $M$ and $q$ at any redshift, so the old average moved cells that belong at $z \sim 0.2$ to Mpc distances.

In [ ]:
CASES = {
    # label: (value given to stalled corners, method)
    "old run_model": (-np.inf, 'legacy'),
    "old, -1 corners": (-1.0, 'legacy'),
    "new": (-1.0, 'weighted'),
}
COLORS = {"old run_model": "C0", "old, -1 corners": "C1", "new": "k"}


def dcom_mpc(z):
    # comoving distance of each cell center, NaN where the cell is dropped (z <= 0)
    good = z > 0.0
    return np.where(good, cosmo.z_to_dcom(np.where(good, z, 1.0)) / MPC, np.nan)


for mod in models.values():
    mod['dc'] = {}
    for case, (stall, method) in CASES.items():
        zin = np.where(mod['redz'] < 0.0, stall, mod['redz'])
        mod['dc'][case] = dcom_mpc(grav.redz_at_bin_centers(zin, method=method, dnum=mod['dnum']))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120, sharey=True)
for ax, (name, mod) in zip(axes, models.items()):
    print(f"{name}: expected binaries (all frequency bins) within")
    for case in ("old, -1 corners", "new"):
        dc = mod['dc'][case].ravel()
        good = np.isfinite(dc)
        idx = np.argsort(dc[good])
        dd, cum = dc[good][idx], np.cumsum(mod['number'].ravel()[good][idx])
        ax.plot(dd, cum, color=COLORS[case], label=case)
        print(f"  {case:>16s}:  10 Mpc {np.interp(10, dd, cum):9.3g},  100 Mpc {np.interp(100, dd, cum):9.3g}")
        if case == "new":
            ref = np.logspace(-3, np.log10(300), 20)
            ax.plot(ref, np.interp(300, dd, cum) * (ref / 300) ** 3, color='0.5', ls='--', label=r'$\propto d_c^3$')
    ax.set(xscale='log', yscale='log', xlim=(1e-3, 1e4), ylim=(1e-3, None),
           xlabel=r'comoving distance $d_c$ [Mpc]', title=name)
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel(r'expected binaries within $d_c$')
axes[0].legend()
plt.show()

## 4. GWB Over Many Realizations

holodeck draws its GWB realizations in Cython (`cyutils.sam_poisson_gwb`), with a generator that `np.random` doesn't seed. Here the Poisson draws are made in the notebook, and every case uses the same draws, since `number` doesn't depend on the redshifts. The differences are therefore the method's alone, with no Monte Carlo noise between the cases. The same draws give each bin's loudest source (Section 5).

Top: median and 16–84% range over realizations. Bottom: new / old for the median (solid) and the 16th and 84th percentiles (dotted). The gray band is the new 16–84% range relative to its median, the realization scatter for scale.

In [ ]:
NREALS, CHUNK = 1000, 50


def realize(mod, seed=165):
    # GWB: hc^2 = sum over cells of hc2 * N, with N ~ Poisson(number), one draw shared by all cases.
    # Loudest source per bin: the drawn cell (N > 0) with the largest hc2.
    rng = np.random.default_rng(seed)
    hc2 = {}
    for case, (stall, method) in CASES.items():
        zin = np.where(mod['redz'] < 0.0, stall, mod['redz'])
        hc2[case] = grav.char_strain_sq_from_bin_edges_redz(mod['edges'], zin, dnum=mod['dnum'], redz_method=method)
        assert np.all(np.isfinite(hc2[case][mod['number'] > 0])), case
    out = {case: dict(gwb=np.zeros((NFREQS, NREALS)), loud=np.zeros((NFREQS, NREALS)),
                      loud_dc=np.zeros((NFREQS, NREALS))) for case in CASES}
    for ff in range(NFREQS):
        sel = mod['number'][..., ff] > 0.0
        num = mod['number'][..., ff][sel]
        h2 = {case: hc2[case][..., ff][sel] for case in CASES}
        dc = {case: mod['dc'][case][..., ff][sel] for case in CASES}
        for r0 in range(0, NREALS, CHUNK):
            draw = rng.poisson(num[:, None], size=(num.size, CHUNK))
            for case in CASES:
                out[case]['gwb'][ff, r0:r0 + CHUNK] = h2[case] @ draw
                # cells not drawn get -1, below any real hc2 (>= 0)
                ii = np.argmax(np.where(draw > 0, h2[case][:, None], -1.0), axis=0)
                out[case]['loud'][ff, r0:r0 + CHUNK] = h2[case][ii]
                out[case]['loud_dc'][ff, r0:r0 + CHUNK] = dc[case][ii]
    for case in CASES:
        out[case]['gwb'] = np.sqrt(out[case]['gwb'])
        out[case]['loud'] = np.sqrt(out[case]['loud'])
    return out


for name, mod in models.items():
    # the per-cell hc2 used here, summed with the expected number, must give holodeck's own expected GWB
    hc2 = grav.char_strain_sq_from_bin_edges_redz(mod['edges'], mod['redz'], dnum=mod['dnum'])
    expect = grav._gws_from_number_grid_integrated_redz(mod['edges'], mod['redz'], mod['number'], False, dnum=mod['dnum'])
    assert np.allclose(np.sqrt(np.sum(hc2 * mod['number'], axis=(0, 1, 2))), expect, rtol=1e-12, atol=0.0)
    mod['reals'] = realize(mod)
    print(f"{name}: {NREALS} realizations done")

In [ ]:
fnhz = fobs_cents * 1e9
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), dpi=120, sharex=True, height_ratios=[2, 1])
for col, (name, mod) in enumerate(models.items()):
    pct = {case: np.percentile(mod['reals'][case]['gwb'], [16, 50, 84], axis=1) for case in CASES}
    ax = axes[0, col]
    for case in CASES:
        ls = '--' if case == "old run_model" else '-'
        ax.plot(fnhz, pct[case][1], color=COLORS[case], ls=ls, label=case)
        ax.fill_between(fnhz, pct[case][0], pct[case][2], color=COLORS[case], alpha=0.15)
    ax.set(xscale='log', yscale='log', title=name)
    ax.grid(True, alpha=0.25)

    ax = axes[1, col]
    new = pct["new"]
    ax.fill_between(fnhz, new[0] / new[1], new[2] / new[1], color='0.5', alpha=0.25, lw=0)
    for case in ("old run_model", "old, -1 corners"):
        ax.plot(fnhz, new[1] / pct[case][1], color=COLORS[case], label=f"new / {case}")
        for qq in (0, 2):
            ax.plot(fnhz, new[qq] / pct[case][qq], color=COLORS[case], ls=':')
    ax.axhline(1.0, color='k', lw=0.8)
    ax.set(xscale='log', yscale='log', ylim=(0.45, 2.5), xlabel='GW frequency [nHz]')
    ax.grid(True, alpha=0.25)
for ax in axes.flat:
    ax.set_xticks([2, 5, 10, 20, 40])
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())
for ax in axes[1]:
    ax.set_yticks([0.5, 0.7, 1.0, 1.4, 2.0])
    ax.yaxis.set_major_formatter(ScalarFormatter())
axes[0, 0].set_ylabel(r'GWB $h_c$')
axes[1, 0].set_ylabel('ratio')
axes[0, 0].legend()
axes[1, 0].legend()
plt.show()

for name, mod in models.items():
    new = np.percentile(mod['reals']['new']['gwb'], [16, 50, 84], axis=1)
    sig = 0.5 * (new[2] - new[0])
    print(f"{name}: realization scatter (16-84 half-width / median) {np.min(sig / new[1]):.1%} to {np.max(sig / new[1]):.1%}")
    for case in ("old run_model", "old, -1 corners"):
        old = np.percentile(mod['reals'][case]['gwb'], [16, 50, 84], axis=1)
        rel = new / old - 1.0
        kk = np.argmax(np.abs(rel[1]))
        print(f"  new vs {case:>16s}: median changes by up to {rel[1, kk]:+.2%} ({rel[1, kk] * old[1, kk] / sig[kk]:+.2f} sigma, "
              f"bin {kk}); 16th/84th by up to {np.max(np.abs(rel[0])):.2%} / {np.max(np.abs(rel[2])):.2%}")

## 5. Loudest Sources

The loudest source in each frequency bin and realization, from the same draws. Single sources (`ss_gws_redz`) always saw the $-1$ corners, in `run_model` too, so the comparison is old, $-1$ corners against new.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120, sharey=True)
bins = np.linspace(-4, 4, 81)
for ax, (name, mod) in zip(axes, models.items()):
    print(f"{name}: loudest source per bin and realization")
    for case in ("old, -1 corners", "new"):
        dc = mod['reals'][case]['loud_dc']
        ax.hist(np.log10(dc.ravel()), bins=bins, histtype='step', color=COLORS[case], lw=1.5, label=case)
        print(f"  {case:>16s}:  within 10 Mpc {np.mean(dc < 10):6.2%},  median d_c {np.median(dc):7.1f} Mpc")
    ratio = mod['reals']['new']['loud'] / mod['reals']["old, -1 corners"]['loud']
    print(f"  new / old loudest h_c, median over bins: {np.median(ratio, axis=1).min():.3f} to {np.median(ratio, axis=1).max():.3f}")
    ax.set(yscale='log', xlabel=r'$\log_{10}$ loudest source $d_c$ [Mpc]', title=name)
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('bins x realizations')
axes[0].legend()
plt.show()

## 6. `'legacy'` Reproduces the Old Average

`old_average` is the loop each call site used before this fix (e.g. `gravwaves.char_strain_sq_from_bin_edges_redz`). `'legacy'` must match it exactly on both kinds of stalled corner, and `'drop'` must match it on the $-\infty$ corners `run_model` made.

This checks the average itself. The end-to-end outputs (`sam.gwb_new`, `sam.gwb`, `lib_tools.run_model`) are compared against the commit before the fix separately.

In [ ]:
def old_average(redz):
    for dd in range(3):
        redz = np.moveaxis(redz, dd, 0)
        redz = kale.utils.midpoints(redz, axis=0)
        redz = np.moveaxis(redz, 0, dd)
    return redz


for name, mod in models.items():
    for stall in (-1.0, -np.inf):
        zin = np.where(mod['redz'] < 0.0, stall, mod['redz'])
        same = np.array_equal(grav.redz_at_bin_centers(zin, method='legacy'), old_average(zin), equal_nan=True)
        print(f"{name}, stalled = {stall}: 'legacy' identical to the old average: {same}")
    zinf = np.where(mod['redz'] < 0.0, -np.inf, mod['redz'])
    same = np.array_equal(grav.redz_at_bin_centers(mod['redz'], method='drop'), old_average(zinf), equal_nan=True)
    print(f"{name}: 'drop' identical to run_model's old average: {same}")

## 7. Summary

Medians over the 1000 realizations; ranges are over frequency bins.

In [ ]:
rows = []
for name, mod in models.items():
    reals = mod['reals']
    med = {case: np.median(reals[case]['gwb'], axis=1) for case in CASES}
    rr = {case: med["new"] / med[case] - 1.0 for case in ("old run_model", "old, -1 corners")}
    near = {case: np.mean(reals[case]['loud_dc'] < 10) for case in ("old, -1 corners", "new")}
    rows.append(f"| {name} | {rr['old run_model'].min():+.1%} to {rr['old run_model'].max():+.1%} "
                f"| {rr['old, -1 corners'].min():+.1%} to {rr['old, -1 corners'].max():+.1%} "
                f"| {near['old, -1 corners']:.2%} | {near['new']:.2%} |")
print("| model | GWB median, new vs old run_model | GWB median, new vs old -1 paths "
      "| loudest within 10 Mpc, old | loudest within 10 Mpc, new |")
print("|---|---|---|---|---|")
print("\n".join(rows))